In [ ]:
dbutils.widgets.text("DBT_SELECT", "vault gold", "dbt --select targets (space-separated)")
dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog catalog")
dbutils.widgets.text("WAREHOUSE_ID", "53165753164ae80e", "SQL Warehouse ID")

In [ ]:
import subprocess, sys

print("Installing dbt-databricks...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "dbt-databricks==1.9.3"],
    capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(f"pip install failed:\n{r.stderr[-1000:]}")
print("dbt-databricks installed")

In [ ]:
import os, pathlib

dbt_select = dbutils.widgets.get("DBT_SELECT")
catalog     = dbutils.widgets.get("CATALOG")
warehouse_id = dbutils.widgets.get("WAREHOUSE_ID")

# Resolve host + token from Databricks runtime context
host  = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
http_path = f"/sql/1.0/warehouses/{warehouse_id}"

# Write a temporary profiles.yml
profiles_dir = pathlib.Path("/tmp/dbt_profiles")
profiles_dir.mkdir(exist_ok=True)
(profiles_dir / "profiles.yml").write_text(f"""cdc_gold:
  target: prod
  outputs:
    prod:
      type: databricks
      host: {host}
      http_path: {http_path}
      token: {token}
      catalog: {catalog}
      schema: gold
      threads: 4
""")
print(f"Profile written to {profiles_dir}/profiles.yml")
print(f"host={host}  http_path={http_path}")

In [ ]:
import sys, pathlib, os

# Locate dbt project (git-source jobs mount at /Repos/.internal/<hash>/...)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
for prefix in ("", "/Workspace"):
    candidate = pathlib.Path(prefix + notebook_path).parent / "dbt_project"
    if candidate.exists():
        project_dir = candidate
        break
else:
    raise FileNotFoundError(f"Cannot find dbt_project relative to {notebook_path!r}")
print(f"dbt project dir: {project_dir}")
print(f"exists: {project_dir.exists()}, contents: {[x.name for x in project_dir.iterdir()][:5]}")

# Run dbt using the Python API (avoids subprocess PATH issues)
sys.path.insert(0, str(project_dir))
from dbt.cli.main import dbtRunner

log_dir = pathlib.Path("/tmp/dbt_logs")
target_dir = pathlib.Path("/tmp/dbt_target")
log_dir.mkdir(exist_ok=True)
target_dir.mkdir(exist_ok=True)

def run_dbt(*args):
    all_args = list(args) + [
        "--profiles-dir", str(profiles_dir),
        "--project-dir", str(project_dir),
        "--log-path", str(log_dir),
        "--target-path", str(target_dir),
    ]
    print(f"\ndbt {' '.join(args)}")
    runner = dbtRunner()
    result = runner.invoke(all_args)
    if result.exception:
        raise RuntimeError(f"dbt {args} failed: {result.exception}") from result.exception
    if not result.success:
        # Extract failure details from the result
        failures = []
        try:
            for res_node in result.result.results:
                if hasattr(res_node, 'status') and str(res_node.status) not in ('success', 'pass', 'warn'):
                    node_name = getattr(res_node, 'node', None)
                    name = getattr(node_name, 'unique_id', str(res_node)) if node_name else str(res_node)
                    msg = getattr(res_node, 'message', '') or ''
                    failures.append(f"  {name}: {msg[:300]}")
        except Exception as e:
            failures.append(f"  (could not extract details: {e})")
        detail = "\n".join(failures[:20]) if failures else "(no detail)"
        raise RuntimeError(f"dbt {args} failures:\n{detail}")
    return result

packages_dir = project_dir / "dbt_packages"
if not packages_dir.exists() or not any(packages_dir.iterdir()):
    print("Running dbt deps...")
    run_dbt("deps")
else:
    print(f"Using pre-committed packages in {packages_dir}")

for target in dbt_select.split():
    run_dbt("build", "--select", target)

print("\ndbt build complete!")
